# E4 — Cross-Window BA Grid

In [1]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [2]:
rows = []
for wk, wm in WINDOWS_META.items():
    if wk == 'W1':
        continue  # W1 already analysed in notebook 13
    for day in wm['days']:
        for sym in [wm['front'], wm['back']]:
            fpath = list(SEAGATE_DIR.glob(f'mbp10_*{sym}*{day}*.parquet'))
            if not fpath:
                continue
            df = pd.read_parquet(fpath[0], columns=['ts_event','bid_px_00','ask_px_00'])
            df['ts_event'] = pd.to_datetime(df['ts_event'])
            rth = ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) >= wm['rth_start_min']) &                   ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) <= wm['rth_end_min'])
            df = df[rth]
            ba = (df['ask_px_00'] - df['bid_px_00']) / TICK
            rows.append(dict(window=wk, symbol=sym, leg='front' if sym==wm['front'] else 'back',
                             day=day, mean_ba=ba.mean()))
            del df; gc.collect()

grid = pd.DataFrame(rows)
fig, axes = plt.subplots(4, 2, figsize=(12, 14), sharey=False)
for row_i, wk in enumerate(['W2','W3','W4','W1']):
    for col_i, leg in enumerate(['front','back']):
        ax = axes[row_i][col_i]
        sub = grid[(grid['window']==wk) & (grid['leg']==leg)]
        if sub.empty:
            ax.set_visible(False); continue
        ax.bar(range(len(sub)), sub['mean_ba'].values, color=WIN_COLORS.get(wk,'grey'), alpha=0.85)
        ax.set_title(f'{wk} {leg}', fontsize=9)
        ax.set_xticks(range(len(sub)))
        ax.set_xticklabels([d[-5:] for d in sub['day'].values], rotation=30, fontsize=7)
        ax.set_ylabel('Mean BA (ticks)', fontsize=8)
fig.suptitle('Cross-Window Mean BA Spread Grid', fontsize=13)
save_fig(fig, 'E4_crosswindow_ba_grid.png')


  Saved --> figures/E4_crosswindow_ba_grid.png
